<a href="https://colab.research.google.com/github/cezmi9104-sys/Open-sourced-off-the-shelf-ion-exchange-membrane/blob/main/Gemma3_12b-son.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ============================================================
# HÜCRE 1 — Ollama Kurulum ve Gemma3 12B Yükleme
# ============================================================

# Önce zstd kur, sonra ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Arka planda başlat
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama çalışıyor!")

# Modeli indir
print("⏳ Gemma3 12B indiriliyor... (8GB, 5-10 dakika sürebilir)")
!ollama pull gemma3:12b
print("✅ Model hazır!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama çalışıyor!
⏳ Gemma3 12B indiriliyor... (8GB, 5-10 dakika sürebilir)

✅ Model hazır!


In [17]:
# ============================================================
# HÜCRE 2 — Çeviri Fonksiyonları
# ============================================================
import requests

def translate_chunk(text: str, chunk_index: int = 0, total_chunks: int = 1) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:12b",
            "prompt": f"Aşağıdaki metni Türkçeye çevir. Sadece çeviriyi yaz, açıklama ekleme.\n\nMETİN:\n{text}\n\nTÜRKÇE ÇEVİRİ:",
            "stream": False,
        },
        timeout=300  # 5 dakika timeout
    )
    result = response.json()["response"].strip()
    print(f"  ✔ Parça {chunk_index + 1}/{total_chunks} çevrildi.")
    return result


def split_text(text: str, max_chars: int = 2500) -> list:
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= max_chars:
            current += ("\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            if len(para) > max_chars:
                for i in range(0, len(para), max_chars):
                    chunks.append(para[i:i + max_chars])
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks


# --- Test ---
print("🧪 Test çevirisi yapılıyor...")
test = translate_chunk("Hello, how are you?", 0, 1)
print(f"Sonuç: {test}")
print("\n📌 Artık Hücre 3'ü çalıştırabilirsiniz!")

🧪 Test çevirisi yapılıyor...
  ✔ Parça 1/1 çevrildi.
Sonuç: Merhaba, nasılsın?

📌 Artık Hücre 3'ü çalıştırabilirsiniz!


In [22]:
# ============================================================
# HÜCRE 3 — PDF Çeviri (OCR Destekli)
# ============================================================

!pip install -q pdfplumber pdf2image pytesseract
!sudo apt-get install -y tesseract-ocr tesseract-ocr-eng poppler-utils

import requests, subprocess, time, pdfplumber, pytesseract
from pdf2image import convert_from_path
from google.colab import files


# --- Ollama kontrol ---
def ollama_baslat():
    try:
        requests.get("http://localhost:11434", timeout=3)
        print("✅ Ollama çalışıyor!")
    except:
        print("⚠️ Ollama başlatılıyor...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(8)
        try:
            requests.get("http://localhost:11434", timeout=3)
            print("✅ Ollama başlatıldı!")
        except:
            print("❌ Hücre 1'i tekrar çalıştır!")
            raise

ollama_baslat()


# --- PDF'den metin çıkar (önce normal, olmadı OCR) ---
def pdf_oku(pdf_path: str) -> str:
    # Önce normal okuma dene
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

    # Metin boşsa OCR kullan
    if len(full_text.strip()) < 100:
        print("⚠️  Normal okuma başarısız, OCR başlatılıyor...")
        pages = convert_from_path(pdf_path, dpi=300)
        for i, page in enumerate(pages):
            print(f"  🔍 Sayfa {i+1}/{len(pages)} OCR işleniyor...")
            text = pytesseract.image_to_string(page, lang="eng")
            full_text += text + "\n"
        print("✅ OCR tamamlandı!")
    else:
        print("✅ Normal metin okuma başarılı!")

    return full_text


# --- PDF Yükle ---
print("\n📂 PDF dosyası seçin:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

full_text = pdf_oku(pdf_path)
print(f"📄 {len(full_text)} karakter okundu.")


# --- Parçalara böl ve çevir ---
chunks = split_text(full_text)
print(f"✂️  {len(chunks)} parçaya bölündü. Çeviri başlıyor...\n")

translated_parts = []
for i, chunk in enumerate(chunks):
    translated = translate_chunk(chunk, i, len(chunks))
    translated_parts.append(translated)

final_translation = "\n\n".join(translated_parts)


# --- Kaydet ve indir ---
output_file = pdf_path.replace(".pdf", "_turkce.txt")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_translation)

print(f"\n✅ Çeviri tamamlandı!")
print(f"📥 İndiriliyor: {output_file}")
files.download(output_file)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (439 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a contro

Saving CN102299326B.pdf to CN102299326B (1).pdf
⚠️  Normal okuma başarısız, OCR başlatılıyor...
  🔍 Sayfa 1/10 OCR işleniyor...
  🔍 Sayfa 2/10 OCR işleniyor...
  🔍 Sayfa 3/10 OCR işleniyor...
  🔍 Sayfa 4/10 OCR işleniyor...
  🔍 Sayfa 5/10 OCR işleniyor...
  🔍 Sayfa 6/10 OCR işleniyor...
  🔍 Sayfa 7/10 OCR işleniyor...
  🔍 Sayfa 8/10 OCR işleniyor...
  🔍 Sayfa 9/10 OCR işleniyor...
  🔍 Sayfa 10/10 OCR işleniyor...
✅ OCR tamamlandı!
📄 20242 karakter okundu.
✂️  6 parçaya bölündü. Çeviri başlıyor...

  ✔ Parça 1/6 çevrildi.
  ✔ Parça 2/6 çevrildi.
  ✔ Parça 3/6 çevrildi.
  ✔ Parça 4/6 çevrildi.
  ✔ Parça 5/6 çevrildi.
  ✔ Parça 6/6 çevrildi.

✅ Çeviri tamamlandı!
📥 İndiriliyor: CN102299326B (1)_turkce.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# HÜCRE 3 — PDF Çeviri (Çince + OCR Destekli)
# ============================================================

!pip install -q pdfplumber pdf2image pytesseract
!sudo apt-get install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-chi-sim poppler-utils
# ↑ chi-sim = Simplified Chinese (Basit Çince)

import requests, subprocess, time, pdfplumber, pytesseract
from pdf2image import convert_from_path
from google.colab import files


# --- Ollama kontrol ---
def ollama_baslat():
    try:
        requests.get("http://localhost:11434", timeout=3)
        print("✅ Ollama çalışıyor!")
    except:
        print("⚠️ Ollama başlatılıyor...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(8)
        try:
            requests.get("http://localhost:11434", timeout=3)
            print("✅ Ollama başlatıldı!")
        except:
            print("❌ Hücre 1'i tekrar çalıştır!")
            raise

ollama_baslat()


# --- PDF dilini otomatik tespit et ---
def dil_tespit(pdf_path):
    """İlk sayfadan dil tahmini yapar"""
    try:
        pages = convert_from_path(pdf_path, dpi=150, last_page=1)
        # Önce Çince dene
        chi_text = pytesseract.image_to_string(pages[0], lang="chi_sim")
        chi_chars = sum(1 for c in chi_text if '\u4e00' <= c <= '\u9fff')
        if chi_chars > 20:
            print("🈳 Çince PDF tespit edildi!")
            return "chi_sim"
        return "eng"
    except:
        return "eng"


# --- PDF'den metin çıkar ---
def pdf_oku(pdf_path: str) -> str:
    # Önce normal okuma dene
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

    if len(full_text.strip()) >= 100:
        print("✅ Normal metin okuma başarılı!")
        return full_text

    # OCR gerekiyor
    print("⚠️  Normal okuma başarısız, OCR başlatılıyor...")
    lang = dil_tespit(pdf_path)
    print(f"🔍 OCR dili: {lang}")

    pages = convert_from_path(pdf_path, dpi=300)
    for i, page in enumerate(pages):
        print(f"  🔍 Sayfa {i+1}/{len(pages)} işleniyor...")
        text = pytesseract.image_to_string(page, lang=lang)
        full_text += text + "\n"

    print("✅ OCR tamamlandı!")
    return full_text


# --- Çeviri fonksiyonu (Çince dahil) ---
def translate_chunk(text: str, chunk_index: int = 0, total_chunks: int = 1) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:12b",
            "prompt": f"Aşağıdaki metni Türkçeye çevir. Metin Çince veya İngilizce olabilir. Sadece çeviriyi yaz, açıklama ekleme.\n\nMETİN:\n{text}\n\nTÜRKÇE ÇEVİRİ:",
            "stream": False,
        },
        timeout=300
    )
    result = response.json()["response"].strip()
    print(f"  ✔ Parça {chunk_index + 1}/{total_chunks} çevrildi.")
    return result


# --- PDF Yükle ---
print("\n📂 PDF dosyası seçin:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

full_text = pdf_oku(pdf_path)
print(f"📄 {len(full_text)} karakter okundu.")

chunks = split_text(full_text)
print(f"✂️  {len(chunks)} parçaya bölündü. Çeviri başlıyor...\n")

translated_parts = []
for i, chunk in enumerate(chunks):
    translated = translate_chunk(chunk, i, len(chunks))
    translated_parts.append(translated)

final_translation = "\n\n".join(translated_parts)

output_file = pdf_path.replace(".pdf", "_turkce.txt")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_translation)

print(f"\n✅ Çeviri tamamlandı!")
print(f"📥 İndiriliyor: {output_file}")
files.download(output_file)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-chi-sim is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Ollama çalışıyor!

📂 PDF dosyası seçin:


Saving review-on-ribbon-silicon-techniques-for-cost-reduction-in-pv-25i1fxjs1r.pdf to review-on-ribbon-silicon-techniques-for-cost-reduction-in-pv-25i1fxjs1r.pdf
✅ Normal metin okuma başarılı!
📄 16600 karakter okundu.
✂️  7 parçaya bölündü. Çeviri başlıyor...

  ✔ Parça 1/7 çevrildi.
  ✔ Parça 2/7 çevrildi.
  ✔ Parça 3/7 çevrildi.
  ✔ Parça 4/7 çevrildi.
  ✔ Parça 5/7 çevrildi.
